# Task 4 : Naïve Bayesian Classifier

## Requirements

 - Given a data set of quiz scores and course results of students in task4_data.csv,
   - #: row index
   - Rank: P → pass, F → Fail
   - Q1, Q2, ..., Q9: quiz scores
 - Scores  are continuous  values.Therefore,  students  propose  an  approach  to discretize  these  values  and  thenmanually  implement  the  Naïve  Bayesian classification algorithm.
 - Finally, compute the accuracy of the classifier in the given data set.
 - Students organize the program regarding to the OOP model, ensure source code is compact and reasonable.
 - Recommendededitor: Google Colab

## Import library

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

## Define Problem Class

In [2]:
class Problem:
    """Class representing the classification problem with data loading and preprocessing."""

    def __init__(self, file_path):
        """Initialize the problem with data from a CSV file.

        Args:
            file_path (str): Path to the CSV file with the data.
        """
        self.file_path = file_path
        self.data = None
        self.discretized_data = None
        self.features = None
        self.target = 'Rank'
        self.classes = None

    def load_data(self):
        """Load data from CSV file."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preprocess_data(self):
        """Preprocess data by removing unnecessary columns and handling missing values."""
        if self.data is None:
            self.load_data()

        # Drop the row index column which is not needed
        if '#' in self.data.columns:
            self.data = self.data.drop('#', axis=1)

        self.data = self.data.loc[:, ~self.data.columns.str.contains('^Unnamed')]
        # Get feature names (all columns except target)
        self.features = [col for col in self.data.columns if col != self.target]

        # Identify unique classes
        self.classes = self.data[self.target].unique()

        return self.data

    def discretize_data(self, bins=4):
      """Discretize continuous quiz scores into equal-width bins.

      Args:
          bins (int): Number of bins to use for discretization.

      Returns:
          pd.DataFrame: DataFrame with discretized values.
      """
      if self.data is None:
          self.preprocess_data()

      self.discretized_data = self.data.copy()

      for feature in self.features:
          min_val = self.data[feature].min()
          max_val = self.data[feature].max()

          # If min = max, assign it all to bin_0
          if min_val == max_val:
              self.discretized_data[feature] = "bin_0"
          else:
            bin_edges = np.linspace(min_val, max_val, bins + 1)
            bin_edges[-1] = max_val * 1.001 + 0.001
            cut_result = pd.cut(
                self.data[feature],
                bins=bin_edges,
                labels=[f"bin_{i}" for i in range(bins)],
                include_lowest=True,
                retbins=True
            )

          self.discretized_data[feature] = cut_result[0]  # Classify
          bin_edges = cut_result[1]
          print(f"{feature} bins:")
          for i in range(len(bin_edges) - 1):
              print(f"  bin_{i}: [{bin_edges[i]:.2f}, {bin_edges[i+1]:.2f})")
      return self.discretized_data

## Define Solution Class

In [ ]:
class Solution:
    """Class implementing the Naïve Bayesian classifier."""

    def __init__(self, problem):
        self.problem = problem
        self.prior_probabilities = {}
        self.conditional_probabilities = defaultdict(lambda: defaultdict(dict))  # No default float assignment
        self.smoothing_factor = 1  # Only use if neccessary

    def train(self, data=None):
        if data is None:
            data = self.problem.discretized_data
        if data is None:
            raise ValueError("No data available for training. Please discretize the data first.")

        # Calculate prior probabilities P(C)
        class_counts = data[self.problem.target].value_counts()
        total_samples = len(data)
        for class_value in self.problem.classes:
            self.prior_probabilities[class_value] = class_counts.get(class_value, 0) / total_samples

        # Calculate conditional probabilities P(F = val | C)
        for class_value in self.problem.classes:
            class_data = data[data[self.problem.target] == class_value]
            class_count = len(class_data)

            for feature in self.problem.features:
                value_counts = class_data[feature].value_counts()
                for feature_value, count in value_counts.items():
                    self.conditional_probabilities[class_value][feature][feature_value] = count / class_count

        return self

    def predict(self, instance):
        probabilities = {}

        for class_value in self.problem.classes:
            prob = self.prior_probabilities.get(class_value, 0)

            for feature in self.problem.features:
                feature_value = instance[feature]

                if feature_value in self.conditional_probabilities[class_value][feature]:
                    prob *= self.conditional_probabilities[class_value][feature][feature_value]
                else:
                    # Use Laplace smoothing if feature_value haven't been seen in this class
                    unique_vals = len(self.conditional_probabilities[class_value][feature])
                    class_count = len(self.problem.data[self.problem.data[self.problem.target] == class_value])
                    prob *= self.smoothing_factor / (class_count + self.smoothing_factor * max(1, unique_vals))

            probabilities[class_value] = prob

        return max(probabilities.items(), key=lambda x: x[1])[0]

    def evaluate(self, data=None):
        if data is None:
            data = self.problem.discretized_data
        if data is None:
            raise ValueError("No data available for evaluation. Please discretize the data first.")

        correct = 0
        for _, row in data.iterrows():
            if self.predict(row) == row[self.problem.target]:
                correct += 1

        return correct / len(data)


In [4]:
def main():
    """Main function to run the Naïve Bayesian classifier."""
    # Initialize problem with data file
    problem = Problem('task4_data.csv')

    # Load and preprocess data
    problem.load_data()
    problem.preprocess_data()

    # Use a fixed number of bins for discretization
    bins = 3  # Using a moderate number of bins
    print(f"Discretizing data with {bins} bins (equal-width bins):\n")
    problem.discretize_data(bins=bins)

    # Initialize and train solution
    solution = Solution(problem)
    solution.train()

    # Evaluate accuracy
    accuracy = solution.evaluate()
    print(f"\nClassification accuracy: {accuracy:.4f}")
    return problem, solution


if __name__ == "__main__":
    main()

Discretizing data with 3 bins (equal-width bins):

Q1 bins:
  bin_0: [0.00, 2.33)
  bin_1: [2.33, 4.67)
  bin_2: [4.67, 7.01)
Q2 bins:
  bin_0: [0.00, 3.33)
  bin_1: [3.33, 6.67)
  bin_2: [6.67, 10.01)
Q3 bins:
  bin_0: [0.00, 3.33)
  bin_1: [3.33, 6.67)
  bin_2: [6.67, 10.01)
Q4 bins:
  bin_0: [0.00, 3.33)
  bin_1: [3.33, 6.67)
  bin_2: [6.67, 10.01)
Q5 bins:
  bin_0: [0.00, 3.00)
  bin_1: [3.00, 6.00)
  bin_2: [6.00, 9.01)
Q6 bins:
  bin_0: [0.00, 3.33)
  bin_1: [3.33, 6.67)
  bin_2: [6.67, 10.01)
Q7 bins:
  bin_0: [0.00, 2.67)
  bin_1: [2.67, 5.33)
  bin_2: [5.33, 8.01)
Q8 bins:
  bin_0: [0.00, 3.10)
  bin_1: [3.10, 6.20)
  bin_2: [6.20, 9.31)
Q9 bins:
  bin_0: [0.00, 3.33)
  bin_1: [3.33, 6.67)
  bin_2: [6.67, 10.01)

Classification accuracy: 0.7603
